###Conv-BiLSTM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn
import plotly.express as px

In [ ]:
#preparing the data for LSTM
df=pd.read_csv("/content/drive/MyDrive/Climat 2001-2018/full_leish_data_2010-2022.csv").set_index("Date")
df.drop(["Annee",'Mois'],axis=1,inplace=True)

In [ ]:
df.head()

,RH2M,TMP_MIN,TMP_MAX,WS2M,WS10M,WD10M,WD2M,UVA,UVB,CLOUD_AMT,SOIL_M,SOIL_W,SOIL_RW,PRECT_TOTAL,SURFACE_P,SH2M,Cases
Date,,,,,,,,,,,,,,,,,
2010-01-01,58.067647,-10.09,25.55,2.966471,4.247059,311.408235,311.491765,8.597647,0.148824,36.217647,0.527647,0.522941,0.534118,40.324706,83.640588,3.928235,71
2010-02-01,67.925294,-5.69,26.46,3.704706,5.248235,274.142353,273.551765,9.837059,0.193529,54.170588,0.591765,0.614706,0.604706,83.442941,83.284706,5.432353,54
2010-03-01,55.660000,-2.11,29.61,3.304118,4.689412,289.415294,288.811765,13.687647,0.314118,42.922941,0.602941,0.616471,0.621765,29.156471,83.697059,4.890000,48
2010-04-01,49.796471,0.37,32.36,2.674118,3.782353,256.629412,250.105882,16.078824,0.385294,47.584706,0.518824,0.485294,0.527059,1.240588,83.464118,5.765882,20
2010-05-01,41.630000,1.15,34.27,3.589412,4.928235,316.550000,313.687647,18.395882,0.453529,32.672353,0.462941,0.368824,0.466471,0.310000,83.564118,4.973529,17


In [ ]:
#preprocessing the data so it's compatible with Conv-BiLSTM architecture
def df_to_X_y(df,variables,window_size=12) :
    df_np=df[variables].values
    X=[]
    y=[]
    for i in range(len(df)-window_size) :
        row=[row for row in df_np[i:(i+window_size-1)]]
        row.append([df_np[i+window_size,0],df_np[i+window_size,1],df_np[i+window_size,2],df_np[i+window_size,3],df_np[i+window_size,4],df_np[(i+window_size-1),-1]])
        label=df_np[i+window_size,-1]
        X.append(row)
        y.append(label)
    return np.array(X),np.array(y)

In [ ]:
X,y=df_to_X_y(df,variables=['RH2M','TMP_MIN','WS2M','SOIL_W','UVB',"Cases"])

In [ ]:
X.shape,y.shape

((144, 12, 6), (144,))

In [ ]:
#splitting the data
X_train,y_train,X_val,y_val=X[:108],y[:108],X[108:],y[108:]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

model = Sequential()
model.add(Conv1D(filters=32, kernel_size=3, activation='linear',input_shape=(12, 6)))
model.add(Bidirectional(LSTM(40,return_sequences=True)))
model.add(Dropout(0.1))
model.add(Bidirectional(LSTM(60)))
model.add(Dense(1))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 10, 32)            608       
                                                                 
 bidirectional (Bidirection  (None, 10, 80)            23360     
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 10, 80)            0         
                                                                 
 bidirectional_1 (Bidirecti  (None, 120)               67680     
 onal)                                                           
                                                                 
 dense (Dense)               (None, 1)                 121       
                                                                 
Total params: 91769 (358.47 KB)
Trainable params: 91769 

In [ ]:
from tensorflow.keras import optimizers
# SGD optimizer
sgd = optimizers.SGD(learning_rate=0.01, momentum=0.9)

# RMSprop optimizer
rmsprop = optimizers.RMSprop(learning_rate=0.02, rho=0.9)

# Adagrad optimizer
adagrad = optimizers.Adagrad(learning_rate=0.01)

# Adadelta optimizer
adadelta = optimizers.Adadelta(learning_rate=2.0, rho=0.99)

# Nadam optimizer
nadam = optimizers.Nadam(learning_rate=0.002, beta_1=0.9, beta_2=0.999)

In [ ]:
#cp=ModelCheckpoint('LSTMs/',save_best_only=True)
cp=ModelCheckpoint('best_Conv-BiLSTM_model.keras',save_best_only=True)
model.compile(loss=MeanSquaredError(),optimizer=Adam(learning_rate=0.02),metrics=[RootMeanSquaredError()])

In [ ]:
model.fit(X_train,y_train,validation_data=(X_val,y_val),callbacks=[cp],epochs=500)

Epoch 1/500
4/4 [==============================] - 0s 93ms/step - loss: 121.6696 - root_mean_squared_error: 11.0304 - val_loss: 349.1752 - val_root_mean_squared_error: 18.6862
Epoch 2/500
4/4 [==============================] - 0s 68ms/step - loss: 96.0886 - root_mean_squared_error: 9.8025 - val_loss: 374.2501 - val_root_mean_squared_error: 19.3455
Epoch 3/500
4/4 [==============================] - 0s 76ms/step - loss: 98.3941 - root_mean_squared_error: 9.9194 - val_loss: 404.7218 - val_root_mean_squared_error: 20.1177
Epoch 4/500
4/4 [==============================] - 0s 59ms/step - loss: 103.9101 - root_mean_squared_error: 10.1936 - val_loss: 356.9457 - val_root_mean_squared_error: 18.8930
Epoch 5/500
4/4 [==============================] - 0s 53ms/step - loss: 96.0788 - root_mean_squared_error: 9.8020 - val_loss: 299.1615 - val_root_mean_squared_error: 17.2963
Epoch 6/500
4/4 [==============================] - 0s 69ms/step - loss: 97.6478 - root_mean_squared_error: 9.8817 - val_loss: 

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
best=load_model("/content/best_Conv-BiLSTM_model.keras")

In [ ]:
val_preds=best.predict(X_val).flatten()
train_preds=best.predict(X_train).flatten()

4/4 [==============================] - 0s 10ms/step


In [ ]:
px.line(convlstm_v,x=convlstm_v.index,y=["V_preds","y_val"])

In [ ]:
convlstm_v=pd.DataFrame({"V_preds" : val_preds, "y_val" : y_val})
convlstm_t=pd.DataFrame({"T_preds" : train_preds, "y_train" : y_train})

In [ ]:
px.line(convlstm_t,x=convlstm_t.index,y=["T_preds","y_train"])

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
print("Test R2 Score :",r2_score(y_val,val_preds))

Test R2 Score : 0.8911448192864994


In [ ]:
print("Training R2 Score :",r2_score(y_train,train_preds))

Train R2 Score : 0.93834252647651


In [ ]:
print("Test MAE Score :",mean_absolute_error(y_val,val_preds))

Test MAE Score : 9.567959394719866


In [ ]:
print("Test RMSE Score :",np.sqrt(mean_squared_error(y_val,val_preds)))

Test RMSE Score : 11.995335360061798


In [ ]:
print("Train MAE Score :",mean_absolute_error(y_train,train_preds))

Train MAE Score : 5.696848849455516


In [ ]:
print("Train RMSE Score :",np.sqrt(mean_squared_error(y_train,train_preds)))

Train RMSE Score : 8.652696222665393


In [ ]:
print("Test MAPE Score :",(mean_absolute_error(y_val, val_preds) / np.mean(y_val)) * 100)

Test MAPE Score : 28.18711441979666


In [ ]:
print("Train MAPE Score :",(mean_absolute_error(y_train, train_preds) / np.mean(y_train)) * 100)

Train MAPE Score : 17.93759987583661
